# Module 4 - Building Segmentation from Aerial Imagery


### Import Libraries


In [ ]:
# Core
import numpy as np

# Geospatial
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape, Polygon
from shapely.ops import unary_union

# ML, AI
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Data viz
import matplotlib.pyplot as plt
import folium
from folium import raster_layers
from pyproj import Transformer
from PIL import Image
from io import BytesIO
import base64

### 4.1. RGB Aerial Image Data from Amsterdam

#### Data Source Details
- **Website**: https://www.beeldmateriaal.nl/dataroom
- **Region**: Amsterdam, Netherlands  
- **Resolution**: 7.5cm per pixel (HRL - High Resolution Layer)
- **Format**: GeoTIFF (1km × 1km tiles)
- **License**: CC BY (Creative Commons Attribution)
- **Downloaded tile**: `2025_148000_468000_RGB_JPEG_hrl.tif`


In [ ]:
# Single file to display
file = '2025_148000_468000_RGB_JPEG_hrl.tif'


# Load image
with rasterio.open(file) as src:
    rgb = src.read([1, 2, 3], out_shape=(3, 2000, 2000))
    rgb = np.transpose(rgb, (1, 2, 0))

# Display
plt.figure(figsize=(7, 7))
plt.imshow(rgb)
plt.title(file, fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

#### Crop the Data

**KEY CONCEPTS:**
- **Window** = rasterio's way to read a spatial subset (x, y, width, height)
- **Transform** = affine matrix mapping pixel coordinates to real-world coordinates


In [ ]:
with rasterio.open('2025_148000_468000_RGB_JPEG_hrl.tif') as src:

    # Define lower-right quadrant window
    w, h = src.width // 2, src.height // 2
    window = Window(w, h, w, h)
    
    # Read RGB and update metadata
    rgb_crop = src.read([1, 2, 3], window=window)
    meta = src.meta.copy()
    meta.update({
        'height': h, 
        'width': w, 
        'transform': src.window_transform(window)  # Update georeference
    })

        
    # Save georeferenced crop
    with rasterio.open('2025_148000_468000_RGB_JPEG_hrl_sample_L.tif', 'w', **meta) as dst:
        dst.write(rgb_crop)
    
    print(f"✓ Cropped and saved: {w}×{h} pixels")
    print(f"  Resolution: {src.res[0]:.4f} m/pixel")


In [ ]:
# Visualize cropped area
rgb_vis = np.transpose(rgb_crop, (1, 2, 0))

plt.figure(figsize=(10, 10))
plt.imshow(rgb_vis)
plt.title('Lower-Right Quadrant (Georeferenced)', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

#### Metadata


In [ ]:
# Print metadata
filepath = '2025_148000_468000_RGB_JPEG_hrl_sample_L.tif'

with rasterio.open(filepath) as src:
    # Basic properties
    print(f"\n📁 File: {filepath}")
    print(f"   Size: {src.width:,} × {src.height:,} pixels")
    print(f"   Bands: {src.count} (RGB)")
    print(f"   Type: {src.dtypes[0]}")
    
    # Resolution
    print(f"\n📏 Resolution: {src.res[0]:.2f} × {src.res[1]:.2f} m/pixel")
    
    # Coverage
    width_m = src.width * src.res[0]
    height_m = src.height * src.res[1]
    area_km2 = (width_m * height_m) / 1_000_000
    
    print(f"\n📐 Coverage:")
    print(f"   {width_m:,.0f} × {height_m:,.0f} meters")
    print(f"   {area_km2:.3f} km²")
    
    # Coordinate system
    print(f"\n🗺️  CRS: {src.crs}")
    print(f"   Bounds: {src.bounds}")

#### Enhanced Visualization

**KEY CONCEPTS:**
- **Percentile stretch** = contrast enhancement mapping 2nd-98th percentile to [0,1]
- **Per-channel normalization** = apply stretch independently to R, G, B


In [ ]:
# Load aerial image
filepath = '2025_148000_468000_RGB_JPEG_hrl_sample_L.tif'

with rasterio.open(filepath) as src:
    rgb = src.read([1, 2, 3])
    rgb = np.transpose(rgb, (1, 2, 0))  # Convert to H,W,C
    resolution = src.res[0]
    

In [ ]:
def enhance_contrast(image):
    """Apply 2-98 percentile stretch for visualization"""
    enhanced = image.astype(np.float32)
    
    # Normalize to [0, 1]
    enhanced = enhanced / 255.0 if enhanced.max() <= 255 else enhanced / 10000.0
    
    # Per-channel percentile stretch
    for c in range(3):
        p2, p98 = np.percentile(enhanced[:, :, c], (2, 98))
        if p98 > p2:
            enhanced[:, :, c] = np.clip((enhanced[:, :, c] - p2) / (p98 - p2), 0, 1)
    
    return enhanced

rgb_enhanced = enhance_contrast(rgb)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original
axes[0].imshow(rgb)
axes[0].set_title('Original\n(Raw pixel values)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Enhanced
axes[1].imshow(rgb_enhanced)
axes[1].set_title('Contrast Enhanced\n(2-98 percentile stretch)', fontsize=12, fontweight='bold')
axes[1].axis('off')


plt.suptitle('Amsterdam Aerial Imagery - 7.5cm Resolution', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2. Create Training Labels

**KEY CONCEPTS:**
- **Canny edge detection** = algorithm that finds sharp intensity changes (building edges)
- **Morphological closing** = fills small gaps between detected edges
- **Contour filling** = converts edge outlines into solid masks
- **Size filtering** = removes objects too small (cars) or too large (parks)


#### Edge Detection

In [ ]:
# Convert to grayscale for edge detection
rgb_uint8 = (rgb_enhanced * 255).astype(np.uint8)
gray = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2GRAY)

# Gaussian blur to reduce noise
gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Canny edge detection (sharp building boundaries)
edges = cv2.Canny(gray_blur, threshold1=70, threshold2=180)

# Morphological closing to fill gaps in edges
kernel = np.ones((3, 3), np.uint8)
edges_closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)


In [ ]:
edges_closed

In [ ]:
# Find and fill contours to create solid building masks
contours, _ = cv2.findContours(edges_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
filled = np.zeros_like(gray, dtype=np.uint8)
cv2.drawContours(filled, contours, -1, color=255, thickness=cv2.FILLED)


####  Size and Shape Filtering

In [ ]:
# Filter by size (200-7500 m²) and shape (aspect ratio < 10)
min_area_px = int(200 / (0.075 ** 2))    
max_area_px = int(7500 / (0.075 ** 2))  

num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(filled, connectivity=8)



building_mask = np.zeros_like(filled, dtype=np.uint8)
buildings_kept = 0

for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]


    # Size filter: realistic building sizes
    if not (min_area_px < area < max_area_px):
        continue
    
    # Shape filter: reject elongated objects (streets)
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    aspect_ratio = max(w, h) / (min(w, h) + 1e-6)
    
    if aspect_ratio < 15:  # Reasonable width/height ratio
        building_mask[labels == i] = 1
        buildings_kept += 1


# Light morphological cleanup
kernel_small = np.ones((3, 3), np.uint8)
building_mask = cv2.morphologyEx(building_mask, cv2.MORPH_OPEN, kernel_small, iterations=1)

print(f"✓ Filtered to {buildings_kept} buildings")
print(f"  Coverage: {building_mask.mean()*100:.1f}% of image")

#### Visualize Results

In [ ]:
# Visualize detection results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original RGB
axes[0].imshow(rgb_enhanced)
axes[0].set_title('Original RGB', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Building mask
axes[1].imshow(building_mask, cmap='binary_r')
axes[1].set_title(f'Building Mask\n({buildings_kept} buildings)', fontsize=12, fontweight='bold')
axes[1].axis('off')


# Overlay on original
axes[2].imshow(rgb_enhanced)
mask_overlay = np.zeros((*building_mask.shape, 4))
mask_overlay[building_mask == 1] = [1, 0, 0, 0.5]
axes[2].imshow(mask_overlay)
axes[2].set_title('Buildings Highlighted', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Edge-Based Building Detection (Training Labels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3. Prepare Training Data with Spatial Split

**KEY CONCEPTS:**
- **Data leakage** = when test data inadvertently contains information from training data
- **Spatial split** = dividing by geography (left/right) instead of random sampling
- **Validation set** = 20% of left-half patches used to monitor training progress
- **Test set** = right half, completely unseen until final evaluation

#### Spatial Split

In [ ]:
# Spatial split to prevent data leakage
h, w = rgb_enhanced.shape[:2]
split_col = w // 2


# Split RGB image
rgb_train = rgb_enhanced[:, :split_col, :]
rgb_test = rgb_enhanced[:, split_col:, :]

# Split building mask
mask_train = building_mask[:, :split_col]
mask_test = building_mask[:, split_col:]


print("✓ Spatial split applied")
print(f"  Training (left):  {rgb_train.shape}")
print(f"  Test (right):     {rgb_test.shape}")

In [ ]:
# Visualize spatial split
fig, axes = plt.subplots(1, 4, figsize=(12, 4))

axes[0].imshow(rgb_train)
axes[0].set_title('Training RGB (LEFT)', fontweight='bold', fontsize=11, color='blue')
axes[0].axis('off')

axes[1].imshow(mask_train, cmap='binary_r')
axes[1].set_title('Training Mask', fontweight='bold', fontsize=11, color='blue')
axes[1].axis('off')

axes[2].imshow(rgb_test)
axes[2].set_title('Test RGB (RIGHT)', fontweight='bold', fontsize=11, color='red')
axes[2].axis('off')

axes[3].imshow(mask_test, cmap='binary_r')
axes[3].set_title('Test Mask (UNSEEN)', fontweight='bold', fontsize=11, color='red')
axes[3].axis('off')

plt.suptitle('Spatial Split: No Geographic Overlap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### Extract Patches from Training Region


**KEY CONCEPTS:**
- **Patch extraction** = breaking large images into manageable 256×256 tiles
- **Overlap stride** = extracting patches with 50% overlap to create more training examples
- **Validation split** = 20% of left-half patches held out to monitor training progress

In [ ]:
# Patch extraction parameters
PATCH_SIZE = 256
STRIDE = 128  # 50% overlap

In [ ]:

def extract_patches(image, mask, patch_size, stride):
    """Extract overlapping patches from one region"""
    patches_img = []
    patches_mask = []
    
    h, w = image.shape[:2]
    
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            img_patch = image[y:y+patch_size, x:x+patch_size]
            mask_patch = mask[y:y+patch_size, x:x+patch_size]
            
            # Keep only patches with buildings (>5% coverage)
            if mask_patch.mean() > 0.05:
                patches_img.append(img_patch)
                patches_mask.append(mask_patch)
    
    return patches_img, patches_mask

In [ ]:
# Extract from training region (left half only)
train_images, train_masks = extract_patches(rgb_train, mask_train, PATCH_SIZE, STRIDE)

print(f"✓ Patches extracted from left half only")
print(f"  Total patches: {len(train_images)}")
print(f"  Patch size: {PATCH_SIZE}×{PATCH_SIZE}, stride: {STRIDE}")
print(f"  Right half untouched — reserved for final testing")

In [ ]:
len(train_images)

#### Create PyTorch Datasets



**KEY CONCEPTS:**
- **Data augmentation** = artificially expanding training data by applying transformations (flips, rotations) that don't change the semantic meaning — a building flipped horizontally is still a building, so this teaches the model to be rotation-invariant without collecting more images
- **Random 80/20 split** = randomly shuffled so validation patches are spread across the whole left half, not spatially clustered
- **Test set** = right half only, evaluated once at the very end

In [ ]:
class BuildingDataset(Dataset):
    """PyTorch Dataset for building segmentation"""
    
    def __init__(self, images, masks, augment=False):
        self.images = images
        self.masks = masks
        self.augment = augment
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx].copy()
        mask = self.masks[idx].copy()
        
        # Random flips for training augmentation
        if self.augment:
            if np.random.rand() > 0.5:
                image = np.fliplr(image).copy()
                mask = np.fliplr(mask).copy()
            if np.random.rand() > 0.5:
                image = np.flipud(image).copy()
                mask = np.flipud(mask).copy()
        
        # Convert to PyTorch tensors (C, H, W format)
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask

# Create full dataset from left-half patches, then random 80/20 split
full_dataset = BuildingDataset(train_images, train_masks, augment=False)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
train_dataset.dataset.augment = True  # Enable augmentation for training only

print(f"✓ Datasets created")
print(f"  Training:   {len(train_dataset)} samples (with augmentation)")
print(f"  Validation: {len(val_dataset)} samples (no augmentation)")
print(f"  Test:       right half — untouched until final evaluation")

#### Create DataLoaders & Visualize


In [ ]:
BATCH_SIZE = 16

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ DataLoaders created")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")


In [ ]:
# Visualize sample batch
sample_images, sample_masks = next(iter(train_loader))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i in range(4):
    img = sample_images[i].numpy().transpose(1, 2, 0)
    msk = sample_masks[i, 0].numpy()
    
    axes[i].imshow(img)
    overlay = np.zeros((*msk.shape, 4))
    overlay[msk == 1] = [1, 0, 0, 0.6]
    axes[i].imshow(overlay)
    axes[i].set_title(f'{msk.mean()*100:.1f}% building', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Sample Training Batch (Left Half Only)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.4. Train U-Net Model


**KEY CONCEPTS:**
- **U-Net architecture** = encoder-decoder network for pixel-wise segmentation
- **BCELoss** = binary cross-entropy loss for binary segmentation (building vs non-building)
- **IoU (Intersection over Union)** = overlap between predicted and true masks / total area covered
- **Epoch** = one complete pass through all training data


#### Define U-Net Architecture

In [ ]:
# Define simplified U-Net (same as Module 2, but 3-channel RGB input)
model = nn.Sequential(
    # Encoder: compress spatial, expand features
    nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 3 RGB channels → 16
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),  # 256×256 → 128×128
    
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),  # 128×128 → 64×64
    
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Conv2d(64, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    
    # Decoder: expand spatial, reduce features
    nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # 64×64 → 128×128
    nn.ReLU(),
    nn.Conv2d(32, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    
    nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),  # 128×128 → 256×256
    nn.ReLU(),
    nn.Conv2d(16, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    
    # Output: building probability per pixel
    nn.Conv2d(16, 1, kernel_size=1),
    nn.Sigmoid()
)

In [ ]:
# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ U-Net created: {total_params:,} parameters")
print(f"  Device: {device}")
print(f"  Input: 256×256 RGB, Output: 256×256 building probability map")

#### Setup Training Components


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#### Define Evaluation Metrics

IoU metric - Intersection over Union

In [ ]:
def calculate_iou(predictions, targets, threshold=0.5):
    """Calculate Intersection over Union for binary segmentation"""
    # Threshold to get binary predictions
    preds = predictions > threshold
    targets = targets.bool()
    
    # Flatten to compute overlap
    preds = preds.view(-1)
    targets = targets.view(-1)
    
    # Intersection and union
    intersection = (preds & targets).float().sum()
    union = (preds | targets).float().sum()
    
    # IoU with smoothing to avoid division by zero
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.item()

print("✓ IoU metric defined")
print("  IoU = Intersection / Union")
print("  Range: 0.0 (no overlap) → 1.0 (perfect)")

#### Training Loop


In [ ]:
NUM_EPOCHS = 20

# Track metrics
history = {
    'train_loss': [],
    'train_iou': [],
    'val_loss': [],
    'val_iou': []
}
best_val_iou = 0.0

print(f"Training for {NUM_EPOCHS} epochs...")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print("="*60)



for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    # === TRAINING ===
    model.train()
    train_loss = 0.0
    train_iou = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_iou += calculate_iou(outputs, masks)
    
    train_loss /= len(train_loader)
    train_iou /= len(train_loader)
    
    # === VALIDATION ===
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            
            val_loss += criterion(outputs, masks).item()
            val_iou += calculate_iou(outputs, masks)
    
    val_loss /= len(val_loader)
    val_iou /= len(val_loader)


    # Store history
    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    
    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), 'best_unet.pth')
        improved = " ⭐ BEST"
    else:
        improved = ""
    
    print(f"  Train: Loss={train_loss:.4f}, IoU={train_iou:.4f}")
    print(f"  Val:   Loss={val_loss:.4f}, IoU={val_iou:.4f}{improved}")

print("\n" + "="*60)
print(f"✓ Training complete! Best validation IoU: {best_val_iou:.4f}")

#### Visualize Training Progress


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# IoU curves
axes[1].plot(history['train_iou'], label='Train', marker='o', linewidth=2)
axes[1].plot(history['val_iou'], label='Validation', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('IoU', fontsize=12)
axes[1].set_title('IoU Score', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f'Training Progress | Best Val IoU: {best_val_iou:.3f}', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5. Test on Full Image


#### Load Best Model

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_unet.pth'))
model.eval()

print(f"✓ Best model loaded (Val IoU: {best_val_iou:.4f})")
print(f"  Testing on full image: {rgb_enhanced.shape[:2]}")

#### Sliding Window Inference

256×256 patches with 128-pixel stride

In [ ]:
# Sliding window parameters
PATCH_SIZE = 256
STRIDE = 128

h, w = rgb_enhanced.shape[:2]
prediction_map = np.zeros((h, w), dtype=np.float32)
count_map = np.zeros((h, w), dtype=np.float32)


with torch.no_grad():
    for y in range(0, h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, STRIDE):
            # Extract patch
            patch = rgb_enhanced[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

            # Convert to tensor
            patch_tensor = torch.from_numpy(patch.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
            
           
            # Predict
            pred = model(patch_tensor).squeeze().cpu().numpy()
            

            # Accumulate
            prediction_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += pred
            count_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += 1

            print(f"  Progress: {(y/(h-PATCH_SIZE))*100:.0f}%", end='\r')

# Average overlapping predictions
prediction_map = prediction_map / (count_map + 1e-6)

#### Calculate Left vs Right IoU


In [ ]:
# Split predictions and ground truth by training region
split_col = w // 2
pred_left = prediction_map[:, :split_col]
pred_right = prediction_map[:, split_col:]
truth_left = mask_train
truth_right = mask_test


def calculate_iou_numpy(pred, truth, threshold=0.5):
    pred_binary = (pred > threshold).astype(bool)
    truth_binary = truth.astype(bool)
    
    intersection = (pred_binary & truth_binary).sum()
    union = (pred_binary | truth_binary).sum()
    
    return intersection / (union + 1e-6)

left_iou = calculate_iou_numpy(pred_left, truth_left)
right_iou = calculate_iou_numpy(pred_right, truth_right)

print("="*60)
print("GENERALIZATION TEST RESULTS")
print("="*60)
print(f"\nLeft Half (TRAINED):       IoU = {left_iou:.4f}")
print(f"Right Half (UNSEEN TEST):  IoU = {right_iou:.4f}")
print(f"Generalization Gap:        {abs(left_iou - right_iou):.4f}")
print("="*60)

####  Visualize Results


In [ ]:
import folium
from folium import raster_layers
from pyproj import Transformer

# Get bounds and transform to lat/lon
with rasterio.open('2025_148000_468000_RGB_JPEG_hrl_sample_L.tif') as src:
    bounds_rd = src.bounds
    transformer = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)
    
    west, south = transformer.transform(bounds_rd.left, bounds_rd.bottom)
    east, north = transformer.transform(bounds_rd.right, bounds_rd.top)
    
    folium_bounds = [[south, west], [north, east]]
    center_lat = (south + north) / 2
    center_lon = (west + east) / 2

In [ ]:
# Create feature groups for each layer
rgb_group = folium.FeatureGroup(name='📷 Aerial Image (7.5cm)', show=True)
truth_group = folium.FeatureGroup(name='🔴 Ground Truth', show=True)
pred_group = folium.FeatureGroup(name='🔵 U-Net Predictions', show=True)


# Create base map (no base tiles - we'll use our own imagery)
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=17,
    control_scale=True,
    tiles=None  # No base map
)


# Add our RGB aerial image
rgb_uint8 = (rgb_enhanced * 255).astype(np.uint8)
folium.raster_layers.ImageOverlay(
    image=rgb_uint8,
    bounds=folium_bounds,
    opacity=1.0
).add_to(rgb_group)



# Add ground truth - red overlay
truth_rgba = np.zeros((*building_mask.shape, 4), dtype=np.uint8)
truth_rgba[building_mask == 1] = [255, 0, 0, 150]
folium.raster_layers.ImageOverlay(
    image=truth_rgba,
    bounds=folium_bounds
).add_to(truth_group)


# Add predictions - striking blue overlay
pred_rgba = np.zeros((*prediction_map.shape, 4), dtype=np.uint8)
pred_rgba[(prediction_map > 0.5)] = [0, 100, 255, 180]
folium.raster_layers.ImageOverlay(
    image=pred_rgba,
    bounds=folium_bounds
).add_to(pred_group)


# Add feature groups to map
rgb_group.add_to(m)
truth_group.add_to(m)
pred_group.add_to(m)

# Layer control (without base map selector)
folium.LayerControl(position='topright', collapsed=False).add_to(m)


m